# 🐍 Day 3: Multiprocessing

- Process
- Pool
- Queue
- Pipe, shared memory

## Why Multiprocessing?

Separate processes bypass the GIL. Use for CPU-bound work. Each process has its own memory.

## Process Basics

Spawn a new process with `multiprocessing.Process`.

In [ ]:
import multiprocessing
import os

def worker():
    print(f"Worker PID: {os.getpid()}")

if __name__ == "__main__":
    p = multiprocessing.Process(target=worker)
    p.start()
    p.join()
    print(f"Main PID: {os.getpid()}")

## if __name__ == "__main__"

Required on Windows (and good practice): prevents child from re-importing and re-running the main block.

## Pool

Process pool for parallel execution. Similar API to ThreadPoolExecutor.

In [ ]:
import multiprocessing

def square(n):
    return n * n

if __name__ == "__main__":
    with multiprocessing.Pool(processes=4) as pool:
        results = pool.map(square, [1, 2, 3, 4, 5])
    print(results)

## Pool: apply_async and get

Async submission and result retrieval.

In [ ]:
import multiprocessing

def cpu_task(n):
    total = 0
    for i in range(n):
        total += i
    return total

if __name__ == "__main__":
    with multiprocessing.Pool(4) as pool:
        async_result = pool.apply_async(cpu_task, (1000000,))
        result = async_result.get()
    print(result)

## Queue

Thread/process-safe queue for passing messages between processes.

In [ ]:
import multiprocessing

def producer(q):
    for i in range(5):
        q.put(i)
    q.put(None)  # Sentinel

def consumer(q):
    while True:
        item = q.get()
        if item is None:
            break
        print(f"Got {item}")

if __name__ == "__main__":
    q = multiprocessing.Queue()
    p1 = multiprocessing.Process(target=producer, args=(q,))
    p2 = multiprocessing.Process(target=consumer, args=(q,))
    p1.start()
    p2.start()
    p1.join()
    p2.join()

## Pipe

Two-way connection between two processes.

In [ ]:
import multiprocessing

def child(conn):
    conn.send("hello from child")
    print("Child received:", conn.recv())
    conn.close()

if __name__ == "__main__":
    parent_conn, child_conn = multiprocessing.Pipe()
    p = multiprocessing.Process(target=child, args=(child_conn,))
    p.start()
    print("Parent received:", parent_conn.recv())
    parent_conn.send("hello from parent")
    p.join()

## Shared Memory (Value, Array)

Share primitive values or arrays between processes. Use with care; locks may be needed.

In [ ]:
import multiprocessing

def increment(shared_val):
    with shared_val.get_lock():
        shared_val.value += 1

if __name__ == "__main__":
    val = multiprocessing.Value('i', 0)
    procs = [multiprocessing.Process(target=increment, args=(val,)) for _ in range(4)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    print(val.value)

## Manager

Create shared objects (list, dict) managed by a server process.

In [ ]:
import multiprocessing

def add_item(shared_list, item):
    shared_list.append(item)

if __name__ == "__main__":
    with multiprocessing.Manager() as manager:
        shared_list = manager.list()
        procs = [multiprocessing.Process(target=add_item, args=(shared_list, i)) for i in range(5)]
        for p in procs:
            p.start()
        for p in procs:
            p.join()
        print(list(shared_list))

## Common Mistakes

- **Forgetting if __name__ == "__main__"**: Required on Windows
- **Pickling**: Arguments must be picklable
- **Shared state without sync**: Use Value.get_lock(), Manager
- **Pool in Jupyter**: May have issues; run as script

## Exercises

In [ ]:
# Exercise 1: Spawn 3 processes that each print their PID.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Use Pool.map to compute factorials of [1,2,3,4,5].
# YOUR CODE HERE

In [ ]:
# Exercise 3: Use Queue to pass numbers from producer to consumer process.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Use Pipe for bidirectional communication between two processes.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Use Manager().dict() to share a dict between processes.
# YOUR CODE HERE

In [ ]:
# Exercise 6: Use Pool.imap for lazy iteration over results.
# YOUR CODE HERE

## Key Takeaways

- Process: spawn; Pool: map/apply_async
- Queue: producer-consumer; Pipe: two-way
- Value/Array: shared primitives; Manager: shared list/dict
- if __name__ == "__main__" on Windows

## 🔜 Next: Day 4 - The GIL

Tomorrow: GIL explanation, when it matters, threads vs processes!